In [51]:
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
import numpy as np
from pathlib import Path

# Path.cwd() gets the current working directory (where your notebook lives)
# .parents[1] moves up two directories (equivalent to '../../')
project_root = Path.cwd().parents[1]

sys.path.insert(0, str(project_root.resolve()))
from src.simulations.GBM import (
    estimate_parameters,
    simulate_paths,
    simulate_paths2,
    simulate_correlated_GBM
)
from src.data.statistics import compute_log_returns
# Assuming returns is your log-return dataframe
AAPL=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\AAPL.parquet")
NVDA=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\NVDA.parquet")
JPM=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\JPM.parquet")
SPY=pd.read_parquet(r"D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\data\raw\SPY.parquet")
prices = pd.DataFrame({
    "AAPL": AAPL["adj_close"],
    "NVDA":NVDA['adj_close'],
    "JPM":JPM["adj_close"],
    "SPY":SPY['adj_close']
})
returns = compute_log_returns(prices)

ImportError: cannot import name 'simulate_correlated_GBM' from 'src.simulations.GBM' (D:\Documentos\Vida_profesional\Coding\Projects\Quant_Risk_Engine\src\simulations\GBM.py)

In [47]:
prices = prices.dropna()
returns = compute_log_returns(prices).dropna()
returns.head()

,AAPL,NVDA,JPM,SPY
1,-0.009770,-0.016135,-0.013284,-0.007601
2,0.007937,0.004185,-0.000796,0.003807
3,-0.004714,0.012034,-0.017147,-0.002816
4,0.015958,0.001874,0.007771,0.005316
5,0.021018,0.010922,0.003645,0.006757


In [46]:
corr_matrix=returns.corr()
type(corr_matrix)

pandas.core.frame.DataFrame

In [35]:
n_assets=4
n_paths=5000
Z = np.random.normal(size=(n_assets, n_paths))
L = np.linalg.cholesky(corr_matrix)
correlated_Z = L @ Z
np.corrcoef(correlated_Z)

array([[1.        , 0.56987075, 0.44529511, 0.7879309 ],
       [0.56987075, 1.        , 0.35287526, 0.70329074],
       [0.44529511, 0.35287526, 1.        , 0.71869346],
       [0.7879309 , 0.70329074, 0.71869346, 1.        ]])

In [36]:
n_assets = 4
n_paths = 5000
n_steps = 252

S0 = prices.iloc[-1].to_numpy()

mu = (returns.mean() * 252).to_numpy()
sigma = (returns.std() * np.sqrt(252)).to_numpy()

paths = np.zeros((n_steps, n_assets, n_paths))

paths[0] = S0[:, None]

dt = 1/252

for t in range(1, n_steps):

    Z = np.random.normal(size=(n_assets, n_paths))
    Z_corr = L @ Z

    paths[t] = (
        paths[t-1]
        * np.exp(
            (mu[:, None] - 0.5 * sigma[:, None]**2) * dt
            + sigma[:, None] * np.sqrt(dt) * Z_corr
        )
    )

In [37]:
print(np.isnan(paths).sum())
print(np.isinf(paths).sum())

0
0


In [40]:
final_returns = np.log(paths[-1] / paths[-2])  # shape (assets, paths)

corr = np.corrcoef(final_returns)
print(corr)

[[1.         0.56944804 0.42547528 0.77981675]
 [0.56944804 1.         0.34932659 0.7118502 ]
 [0.42547528 0.34932659 1.         0.70672031]
 [0.77981675 0.7118502  0.70672031 1.        ]]
